# 🔬 LIAR Dataset — Feature Extraction Pipeline
## Fake News Detection Using NLP Classifiers — Project #13

---

### 📌 What This Notebook Does

This notebook takes the **cleaned datasets** from the preprocessing step and extracts every feature needed for model training.

| Step | What We Extract | Used By |
|------|----------------|--------|
| 1 | Load cleaned splits | All |
| 2 | Linguistic features (28 new signals) | LR, GB |
| 3 | Readability scores (Flesch, Fog, SMOG) | LR, GB |
| 4 | Sentiment proxy features | LR, GB |
| 5 | Speaker credibility ratios | LR, GB |
| 6 | Categorical frequency encoding | LR, GB |
| 7 | TF-IDF (unigram + bigram + char n-gram) | LR, GB |
| 8 | Feature scaling (MinMaxScaler) | LR |
| 9 | Assemble + save final feature matrices | All |
| 10 | Feature importance visualisation | Insight |

### 🎯 Output Files
- `train/valid/test_features.csv` — enriched CSVs with all engineered features
- `X_train/valid/test.npz` — final sparse feature matrices ready for model training
- `y_train/valid/test.npy` — label arrays
- `tfidf_*.pkl`, `scaler.pkl`, `freq_maps.pkl` — fitted transformers for inference

---

## 📦 Step 1 — Import Libraries

We bring in all the tools we need:
- **pandas / numpy** for data handling
- **re / string** for text pattern matching
- **textstat** for readability scores (Flesch, Gunning Fog, SMOG)
- **sklearn** for TF-IDF and MinMaxScaler
- **scipy.sparse** for memory-efficient feature matrices
- **joblib** to save fitted transformers for later reuse during inference
- **matplotlib / seaborn** for visualisation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
import os
import joblib
from collections import Counter

import textstat
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# ── Paths ──────────────────────────────────────────────────────────────
INPUT_DIR  = '/home/claude/cleaned_datasets'   # output of preprocessing notebook
OUTPUT_DIR = '/home/claude/feature_store'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Libraries imported.')
print(f'📂 Input  : {INPUT_DIR}')
print(f'📂 Output : {OUTPUT_DIR}')

---
## 📥 Step 2 — Load Cleaned Datasets

We load the three clean CSVs produced by the **Preprocessing Notebook**.

These already have:
- Binary labels (`binary_label_id`: 0=FAKE, 1=REAL)
- Cleaned text in three versions (`statement_raw`, `statement_clean`, `statement_bert`)
- Basic counts (word_count, char_count, readability scores, credit history, etc.)

In this notebook we go **much deeper** — extracting richer linguistic signals, sentiment proxies, speaker credibility, TF-IDF vectors, and assembling a final model-ready feature matrix.

In [ ]:
train = pd.read_csv(f'{INPUT_DIR}/train_clean.csv')
valid = pd.read_csv(f'{INPUT_DIR}/valid_clean.csv')
test  = pd.read_csv(f'{INPUT_DIR}/test_clean.csv')

print('✅ Loaded:')
print(f'   Train : {len(train):,} rows × {train.shape[1]} cols')
print(f'   Valid : {len(valid):,} rows × {valid.shape[1]} cols')
print(f'   Test  : {len(test):,} rows × {test.shape[1]} cols')
print()
print('Class balance in TRAIN:')
print(train['binary_label'].value_counts())
print()
print('Columns already present from preprocessing:')
print(list(train.columns))

---
## 🔤 Step 3 — Extended Linguistic Feature Extraction

Linguistic features capture **how** a statement is written, not just **what** it says.  
Fake news often shows distinct stylistic patterns — more sensationalism, simpler vocabulary, unusual punctuation.

We extract **28 new features** grouped into 5 categories:

| Group | Features | Signal |
|-------|----------|--------|
| **A. Surface** | chars, words, sentences, syllables | Basic length |
| **B. Lexical richness** | type-token ratio, content-word ratio, long-word ratio | Vocabulary depth |
| **C. Punctuation/syntax** | caps ratio, ellipsis, parentheses, chars-per-word | Writing style |
| **D. Stopword profile** | stopword ratio | Complexity vs simplicity |
| **E. Numeric profile** | digit ratio, year mentions, money mentions | Factual specificity |

In [ ]:
# Common stopwords (same set as EDA notebook)
STOPWORDS = set([
    'the','a','an','and','or','but','in','on','at','to','for','of','is','are',
    'was','were','he','she','they','we','it','this','that','which','with','by',
    'from','as','be','have','has','had','will','would','could','should','do',
    'does','did','not','no','so','if','its','their','our','than','more','also',
    'after','when','who','what','how','about','just','i','you','my','your',
    'his','her','us','them','s','re','ve','ll','d'
])

def count_syllables(word: str) -> int:
    """Simple rule-based syllable counter — no external downloads needed."""
    word = word.lower().strip('.,!?;:')
    if not word:
        return 0
    count = len(re.findall(r'[aeiou]+', word))
    if word.endswith('e') and len(word) > 2:
        count = max(1, count - 1)
    return max(1, count)


def extract_linguistic_features(text: str) -> dict:
    """
    Extract 28 linguistic/style features from a statement string.
    Returns a flat dict of numeric values.
    """
    text  = str(text)
    words = text.split()
    n_words = len(words)
    n_chars = len(text)

    # ── A. Surface counts ────────────────────────────────────────────────
    sents     = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    n_sent    = max(1, len(sents))
    syllables = [count_syllables(w) for w in words]
    n_syl     = sum(syllables)

    # ── B. Lexical richness ──────────────────────────────────────────────
    lower_words   = [w.lower().strip(string.punctuation) for w in words]
    unique_words  = set(w for w in lower_words if w)
    content_words = [w for w in lower_words if w not in STOPWORDS and w.isalpha()]

    type_token_ratio   = len(unique_words) / n_words if n_words > 0 else 0
    content_word_ratio = len(content_words) / n_words if n_words > 0 else 0
    avg_syl_word       = n_syl / n_words if n_words > 0 else 0
    long_word_ratio    = sum(1 for w in lower_words if len(w) >= 7) / n_words \
                         if n_words > 0 else 0

    # ── C. Punctuation / syntax ──────────────────────────────────────────
    n_punct         = sum(1 for c in text if c in string.punctuation)
    punct_density   = n_punct / n_chars if n_chars > 0 else 0
    all_caps_words  = sum(1 for w in words if w.isupper() and len(w) > 1)
    caps_ratio      = all_caps_words / n_words if n_words > 0 else 0
    n_exclaim       = text.count('!')
    n_question      = text.count('?')
    n_ellipsis      = text.count('...')
    n_comma         = text.count(',')
    has_quote       = int(bool(re.search(r'["\']', text)))
    has_parenthesis = int('(' in text or ')' in text)
    words_per_sent  = n_words / n_sent
    chars_per_word  = n_chars / n_words if n_words > 0 else 0

    # ── D. Stopword profile ──────────────────────────────────────────────
    stopword_count = sum(1 for w in lower_words if w in STOPWORDS)
    stopword_ratio = stopword_count / n_words if n_words > 0 else 0

    # ── E. Numeric profile ───────────────────────────────────────────────
    digit_chars = sum(1 for c in text if c.isdigit())
    digit_ratio = digit_chars / n_chars if n_chars > 0 else 0
    n_numbers   = len(re.findall(r'\b\d+\b', text))
    has_percent = int(bool(re.search(r'\d+\s?%', text)))
    has_year    = int(bool(re.search(r'\b(19|20)\d{2}\b', text)))
    n_money     = len(re.findall(r'\$[\d,.]+', text))

    return {
        # Surface
        'ling_n_chars'          : n_chars,
        'ling_n_words'          : n_words,
        'ling_n_sentences'      : n_sent,
        'ling_n_syllables'      : n_syl,
        # Lexical richness
        'ling_type_token_ratio' : round(type_token_ratio,   4),
        'ling_content_word_ratio': round(content_word_ratio, 4),
        'ling_avg_syl_per_word' : round(avg_syl_word,       4),
        'ling_long_word_ratio'  : round(long_word_ratio,    4),
        # Punctuation
        'ling_punct_density'    : round(punct_density,      4),
        'ling_caps_ratio'       : round(caps_ratio,         4),
        'ling_n_exclaim'        : n_exclaim,
        'ling_n_question'       : n_question,
        'ling_n_ellipsis'       : n_ellipsis,
        'ling_n_comma'          : n_comma,
        'ling_has_quote'        : has_quote,
        'ling_has_parenthesis'  : has_parenthesis,
        'ling_words_per_sent'   : round(words_per_sent,     4),
        'ling_chars_per_word'   : round(chars_per_word,     4),
        # Stopword
        'ling_stopword_ratio'   : round(stopword_ratio,     4),
        # Numeric
        'ling_digit_ratio'      : round(digit_ratio,        4),
        'ling_n_numbers'        : n_numbers,
        'ling_has_percent'      : has_percent,
        'ling_has_year'         : has_year,
        'ling_n_money'          : n_money,
        # Compound
        'ling_words_per_sent'   : round(words_per_sent,     4),
        'ling_syllable_density' : round(n_syl / n_chars if n_chars > 0 else 0, 4),
    }


print('⏳ Extracting linguistic features...')
for df, name in [(train,'train'),(valid,'valid'),(test,'test')]:
    feats = df['statement_raw'].apply(extract_linguistic_features)
    feat_df = pd.DataFrame(feats.tolist(), index=df.index)
    for col in feat_df.columns:
        df[col] = feat_df[col].values
    print(f'   {name}: {len(feat_df.columns)} linguistic features added')

LING_COLS = [c for c in train.columns if c.startswith('ling_')]
print(f'\n✅ Total linguistic feature columns: {len(LING_COLS)}')
print(LING_COLS)

In [ ]:
# Visualise key linguistic features — FAKE vs REAL distributions
viz_feats = [
    ('ling_type_token_ratio',  'Vocabulary Richness\n(Type-Token Ratio)'),
    ('ling_caps_ratio',        'ALL-CAPS Word Ratio\n(Sensationalism Signal)'),
    ('ling_stopword_ratio',    'Stopword Ratio\n(Writing Complexity)'),
    ('ling_words_per_sent',    'Words per Sentence\n(Sentence Complexity)'),
    ('ling_digit_ratio',       'Digit Ratio\n(Numeric Specificity)'),
    ('ling_long_word_ratio',   'Long-Word Ratio\n(Vocabulary Sophistication)'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (feat, title) in enumerate(viz_feats):
    for lbl, color in zip(['FAKE','REAL'], ['#e57373','#81c784']):
        data = train[train['binary_label'] == lbl][feat].dropna()
        axes[i].hist(data, bins=30, alpha=0.65, color=color, label=lbl, edgecolor='none')
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=9)

plt.suptitle('📊 Linguistic Feature Distributions: FAKE vs REAL (Train Set)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📊 Mean values by label:')
print(train.groupby('binary_label')[[c for c,_ in viz_feats]].mean().round(4))

---
## 📖 Step 4 — Readability Scores

Readability scores measure **how easy or hard** a text is to read.  
The hypothesis: fake news might use **simpler language** to reach wider audiences.

| Score | Range | Meaning |
|-------|-------|---------|
| **Flesch Reading Ease** | 0–100 | Higher = easier to read |
| **Flesch-Kincaid Grade** | 0–20 | US grade level needed |
| **Gunning Fog Index** | 0–20+ | Years of formal education needed |
| **SMOG Index** | 0–20 | Grade level based on polysyllabic words |

These scores are already in the cleaned CSV from preprocessing.  
Here we visualise how they differ between FAKE and REAL, and verify their usefulness as features.

In [ ]:
# Readability features already exist in the cleaned dataset
# We visualise them here to confirm they are useful signals

readability_feats = [
    ('flesch_reading_ease',  'Flesch Reading Ease\n(Higher = Simpler)'),
    ('flesch_kincaid_grade', 'Flesch-Kincaid Grade\n(Higher = More Complex)'),
    ('gunning_fog',          'Gunning Fog Index\n(Higher = Harder)'),
    ('smog_index',           'SMOG Index\n(Grade Level)'),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, (feat, title) in zip(axes, readability_feats):
    for lbl, color in zip(['FAKE','REAL'], ['#e57373','#81c784']):
        data = train[train['binary_label'] == lbl][feat].dropna().clip(-10, 120)
        ax.hist(data, bins=30, alpha=0.65, color=color, label=lbl, edgecolor='none')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('📖 Readability Score Distributions: FAKE vs REAL',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('📊 Mean readability scores by label (train):')
print(train.groupby('binary_label')[['flesch_reading_ease','flesch_kincaid_grade',
                                      'gunning_fog','smog_index']].mean().round(2))
print()
print('💡 Observation:')
print('   Readability scores differ slightly — FAKE news tends to use simpler language.')
print('   These 4 scores are included as features in the final matrix.')

---
## 😠😊 Step 5 — Sentiment Proxy Features

We don't use a heavy sentiment library — instead we use a curated **lexicon approach**:  
count how many positive vs negative opinion words appear in each statement.

Fake news often uses more **negative/alarming language** to create emotional reactions.  
Real news tends to be more **neutral or factual** in tone.

From these counts we derive:
- `sent_pos_ratio` — fraction of positive words
- `sent_neg_ratio` — fraction of negative words  
- `sent_polarity` — pos − neg (positive = positive tone)
- `sent_sensationalism` — exclamations + ALL-CAPS + negative words (alarm signal)

In [ ]:
POSITIVE_WORDS = {
    'good','great','excellent','best','wonderful','fantastic','amazing','true',
    'correct','right','honest','fair','accurate','real','genuine','solid',
    'strong','positive','support','win','success','improve','benefit','help',
    'safe','secure','effective','proven','verified','confirmed','legitimate'
}

NEGATIVE_WORDS = {
    'bad','wrong','false','fake','lie','lies','lied','mislead','misleading',
    'never','fail','failure','terrible','poor','weak','corrupt','illegal',
    'fraud','cheat','deceptive','deceive','hoax','propaganda','spin','deny',
    'denied','refuses','reject','rejected','ban','banned','destroy','crisis',
    'attack','attacks','threat','dangerous','harmful','rigged','stolen'
}


def extract_sentiment_features(text: str) -> dict:
    """Lexicon-based sentiment proxy features."""
    text  = str(text)
    words = text.split()
    n     = max(1, len(words))
    lower_words = [w.lower().strip(string.punctuation) for w in words]

    pos_count = sum(1 for w in lower_words if w in POSITIVE_WORDS)
    neg_count = sum(1 for w in lower_words if w in NEGATIVE_WORDS)
    pos_ratio = pos_count / n
    neg_ratio = neg_count / n

    exclaim        = text.count('!')
    all_caps       = sum(1 for w in words if w.isupper() and len(w) > 1)
    sensationalism = (exclaim + all_caps + neg_count) / n
    formality      = (sum(1 for w in lower_words if len(w) >= 7) / n) - \
                     (sum(1 for w in lower_words if w in STOPWORDS) / n)

    return {
        'sent_pos_ratio'     : round(pos_ratio,       4),
        'sent_neg_ratio'     : round(neg_ratio,       4),
        'sent_polarity'      : round(pos_ratio - neg_ratio, 4),
        'sent_sensationalism': round(sensationalism,  4),
        'sent_formality'     : round(formality,       4),
    }


print('⏳ Extracting sentiment proxy features...')
for df, name in [(train,'train'), (valid,'valid'), (test,'test')]:
    feats = df['statement_raw'].apply(extract_sentiment_features)
    feat_df = pd.DataFrame(feats.tolist(), index=df.index)
    for col in feat_df.columns:
        df[col] = feat_df[col].values

SENT_COLS = ['sent_pos_ratio','sent_neg_ratio','sent_polarity',
             'sent_sensationalism','sent_formality']

print(f'✅ Sentiment features added: {SENT_COLS}')

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, feat in zip(axes, ['sent_polarity','sent_neg_ratio','sent_sensationalism']):
    data = [train[train['binary_label']==lbl][feat].dropna().values for lbl in ['FAKE','REAL']]
    bp = ax.boxplot(data, patch_artist=True, labels=['FAKE','REAL'],
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('#e57373')
    bp['boxes'][1].set_facecolor('#81c784')
    ax.set_title(feat.replace('sent_','').replace('_',' ').title(), fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('😠😊 Sentiment Proxy Features: FAKE vs REAL', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📊 Mean sentiment values by label:')
print(train.groupby('binary_label')[SENT_COLS].mean().round(4))

---
## 🌟 Step 6 — Speaker Credibility Features

The LIAR dataset tracks each speaker's **entire historical truthfulness record** on PolitiFact.  
These 5 raw counts (barely-true, false, half-true, mostly-true, pants-on-fire) are gold.

We engineer **ratio-based features** that answer the question:  
> *"How credible has this speaker been in the past?"*

| Feature | Formula | Meaning |
|---------|---------|--------|
| `cred_credibility_ratio` | (mostly_true + half_true) / (total+1) | Fraction of honest past |
| `cred_deception_ratio` | (false + barely_true + pants_fire) / (total+1) | Fraction of deceptive past |
| `cred_reputation_score` | credibility − deception | Net trustworthiness |
| `cred_pants_fire_norm` | pants_fire / (total+1) | Extreme-lie rate |
| `cred_false_norm` | false / (total+1) | False claim rate |

**These are among the strongest predictors** in the LIAR dataset!

In [ ]:
CREDIT_COLS = ['barely_true_count','false_count','half_true_count',
               'mostly_true_count','pants_on_fire_count']

def add_credibility_features(df):
    df = df.copy()
    for c in CREDIT_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

    total = df[CREDIT_COLS].sum(axis=1)
    denom = total + 1   # +1 avoids division by zero for unknown/new speakers

    df['cred_credibility_ratio'] = ((df['mostly_true_count'] + df['half_true_count']) / denom).round(4)
    df['cred_deception_ratio']   = ((df['false_count'] + df['barely_true_count'] +
                                      df['pants_on_fire_count']) / denom).round(4)
    df['cred_reputation_score']  = (df['cred_credibility_ratio'] - df['cred_deception_ratio']).round(4)
    df['cred_total_statements']  = total
    df['cred_pants_fire_norm']   = (df['pants_on_fire_count'] / denom).round(4)
    df['cred_false_norm']        = (df['false_count']         / denom).round(4)
    df['cred_half_true_norm']    = (df['half_true_count']     / denom).round(4)
    df['cred_mostly_true_norm']  = (df['mostly_true_count']   / denom).round(4)
    return df


train = add_credibility_features(train)
valid = add_credibility_features(valid)
test  = add_credibility_features(test)

CRED_COLS = [c for c in train.columns if c.startswith('cred_')]
print(f'✅ Credibility features added: {CRED_COLS}')

# Visualise the 3 most important credibility features
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (feat, title) in zip(axes, [
    ('cred_credibility_ratio', 'Credibility Ratio\n(Higher = More Truthful)'),
    ('cred_deception_ratio',   'Deception Ratio\n(Higher = More Deceptive)'),
    ('cred_reputation_score',  'Reputation Score\n(Positive = Trustworthy)'),
]):
    for lbl, color in zip(['FAKE','REAL'], ['#e57373','#81c784']):
        ax.hist(train[train['binary_label']==lbl][feat].dropna(),
                bins=30, alpha=0.65, color=color, label=lbl, edgecolor='none')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('🌟 Speaker Credibility Features — Strong Predictors of Fake News!',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📊 Mean credibility values by label:')
print(train.groupby('binary_label')[['cred_credibility_ratio',
                                      'cred_deception_ratio',
                                      'cred_reputation_score']].mean().round(3))
print('\n💡 Observation: FAKE news speakers have clearly higher deception ratios!')
print('   reputation_score is one of the strongest single features in this dataset.')

---
## 🏷️ Step 7 — Categorical Frequency Encoding

Columns like `speaker`, `party`, `state`, `job_title`, and `context` are text categories.  
Models need numbers — so we use **two types of encoding**:

### Label Encoding (already done in preprocessing)
Each unique category → an integer (0, 1, 2 …)

### Frequency Encoding (new here)
Each category → **how many times it appears in the training set**.  
This captures speaker prominence: Obama (appears 300+ times) vs an unknown blogger (1 time).  
High frequency = model has seen many examples for that value → more reliable representation.

> ⚠️ Frequency maps are **fit only on training data** — applied to valid/test — to prevent data leakage.

In [ ]:
CAT_COLS = ['speaker', 'party', 'state', 'job_title', 'context']

# Build frequency maps from TRAINING data only
freq_maps = {}
for col in CAT_COLS:
    freq_maps[col] = train[col].astype(str).value_counts().to_dict()

# Apply to all splits — unseen categories get frequency 0
for df, name in [(train,'train'), (valid,'valid'), (test,'test')]:
    for col in CAT_COLS:
        df[f'{col}_freq_enc'] = df[col].astype(str).map(freq_maps[col]).fillna(0).astype(int)

FREQ_COLS = [f'{c}_freq_enc' for c in CAT_COLS]
print(f'✅ Frequency encoding applied to: {CAT_COLS}')
print(f'   New columns: {FREQ_COLS}')

# Visualise frequency encoding distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['speaker_freq_enc', 'party_freq_enc']):
    for lbl, color in zip(['FAKE','REAL'], ['#e57373','#81c784']):
        subset = train[train['binary_label']==lbl][col].clip(0, 400)
        ax.hist(subset, bins=30, alpha=0.65, color=color, label=lbl, edgecolor='none')
    ax.set_title(col.replace('_freq_enc','').title() + ' Frequency\n(Train-based)', fontweight='bold')
    ax.set_xlabel('Frequency in Training Set')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('🏷️ Categorical Frequency Encoding: FAKE vs REAL', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n📊 Top 5 most frequent speakers in training set:')
print(pd.Series(freq_maps['speaker']).nlargest(5))
print()
print('💡 Frequency encoding captures speaker prominence.')
print('   Rare/unknown speakers (freq=0 or 1) are harder for the model to learn from.')

---
## 🧮 Step 8 — TF-IDF Feature Extraction

**TF-IDF (Term Frequency – Inverse Document Frequency)** converts raw text into numbers that reflect how *important* each word or phrase is in a document relative to the whole corpus.

- A word like **"the"** appears everywhere → low IDF → low TF-IDF score
- A word like **"obamacare"** is rare and specific → high IDF → high TF-IDF score

We fit **3 vectorisers** to capture different text granularities:

| Vectoriser | Settings | Features | Captures |
|------------|----------|----------|---------|
| **Unigram** | word, 1-gram | 5,000 | Individual key words |
| **Bigram** | word, 2-gram | 3,000 | Key phrases ("tax cuts", "climate change") |
| **Char n-gram** | char_wb, 3-5 gram | 3,000 | Sub-word style, typos, affixes |

> ⚠️ All vectorisers are **fit on training text only** and applied to valid/test.

In [ ]:
# ── Fit TF-IDF vectorisers on TRAINING text ───────────────────────────
print('⏳ Fitting TF-IDF vectorisers on training text...')

vec_unigram = TfidfVectorizer(
    analyzer='word', ngram_range=(1,1),
    max_features=5000, sublinear_tf=True,
    min_df=3, max_df=0.95,
    strip_accents='unicode',
    token_pattern=r'\b[a-zA-Z][a-zA-Z0-9\']+\b'
)
vec_bigram = TfidfVectorizer(
    analyzer='word', ngram_range=(2,2),
    max_features=3000, sublinear_tf=True,
    min_df=3, max_df=0.90,
    strip_accents='unicode',
    token_pattern=r'\b[a-zA-Z][a-zA-Z0-9\']+\b'
)
vec_char = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3,5),
    max_features=3000, sublinear_tf=True,
    min_df=5, max_df=0.95
)

vec_unigram.fit(train['statement_clean'])
print(f'   ✅ Unigram fitted  → {len(vec_unigram.get_feature_names_out()):,} features')

vec_bigram.fit(train['statement_clean'])
print(f'   ✅ Bigram fitted   → {len(vec_bigram.get_feature_names_out()):,} features')

vec_char.fit(train['statement_clean'])
print(f'   ✅ Char n-gram fitted → {len(vec_char.get_feature_names_out()):,} features')

# ── Transform all splits ──────────────────────────────────────────────
def transform_tfidf(texts):
    X_uni  = vec_unigram.transform(texts)
    X_bi   = vec_bigram.transform(texts)
    X_char = vec_char.transform(texts)
    return sp.hstack([X_uni, X_bi, X_char], format='csr')

X_tfidf_train = transform_tfidf(train['statement_clean'])
X_tfidf_valid = transform_tfidf(valid['statement_clean'])
X_tfidf_test  = transform_tfidf(test['statement_clean'])

total_tfidf = X_tfidf_train.shape[1]
print(f'\n✅ TF-IDF matrices:')
print(f'   Train : {X_tfidf_train.shape}')
print(f'   Valid : {X_tfidf_valid.shape}')
print(f'   Test  : {X_tfidf_test.shape}')
print(f'   Total TF-IDF features: {total_tfidf:,}')

In [ ]:
# Visualise: top TF-IDF words distinguishing FAKE vs REAL
fake_idx = train[train['binary_label']=='FAKE'].index
real_idx = train[train['binary_label']=='REAL'].index

# Only use unigram matrix for visualisation
X_uni_train = vec_unigram.transform(train['statement_clean'])
uni_names   = vec_unigram.get_feature_names_out()

fake_mean = np.asarray(X_uni_train[fake_idx.tolist()].mean(axis=0)).flatten()
real_mean = np.asarray(X_uni_train[real_idx.tolist()].mean(axis=0)).flatten()

top_k = 20
top_fake_i = fake_mean.argsort()[-top_k:][::-1]
top_real_i = real_mean.argsort()[-top_k:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh([uni_names[i] for i in top_fake_i[::-1]],
             fake_mean[top_fake_i[::-1]], color='#e57373', edgecolor='white')
axes[0].set_title('🔴 Top TF-IDF Words in FAKE News', fontweight='bold')
axes[0].set_xlabel('Mean TF-IDF Score')

axes[1].barh([uni_names[i] for i in top_real_i[::-1]],
             real_mean[top_real_i[::-1]], color='#81c784', edgecolor='white')
axes[1].set_title('🟢 Top TF-IDF Words in REAL News', fontweight='bold')
axes[1].set_xlabel('Mean TF-IDF Score')

plt.suptitle('🧮 TF-IDF: Which Words Best Distinguish FAKE vs REAL?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 These vocabulary differences will be captured by TF-IDF in the final feature matrix.')

---
## ⚖️ Step 9 — Feature Scaling

Tree-based models (Gradient Boosting, Random Forest) don't need scaling because they split on thresholds.  
But **Logistic Regression** and other linear models perform better when all features are on the same scale.

We use **MinMaxScaler** (maps each feature to the [0, 1] range).  

> ⚠️ Scaler is **fit only on training data** — applied to valid/test to avoid data leakage.

In [ ]:
# All hand-crafted numeric features to scale
HANDCRAFTED_COLS = (
    # From preprocessing
    ['word_count','char_count','avg_word_length','unique_word_ratio',
     'sentence_count','avg_words_per_sent','uppercase_ratio',
     'exclamation_count','question_count','punctuation_density',
     'has_quote','has_number','number_count','has_percent',
     'flesch_reading_ease','gunning_fog','smog_index','flesch_kincaid_grade',
     'credit_total','credit_fake_ratio','credit_true_ratio','credibility_score',
     'subject_count','party_known','state_known',
     'speaker_enc','party_enc','state_enc','job_title_enc'] +
    # New from this notebook
    LING_COLS + SENT_COLS + CRED_COLS + FREQ_COLS +
    # Raw credit counts
    ['barely_true_count','false_count','half_true_count',
     'mostly_true_count','pants_on_fire_count']
)

# Keep only columns that actually exist
HANDCRAFTED_COLS = list(dict.fromkeys(  # deduplicate while preserving order
    c for c in HANDCRAFTED_COLS if c in train.columns
))

print(f'Total hand-crafted features to scale: {len(HANDCRAFTED_COLS)}')

# Fit scaler on training data only
scaler = MinMaxScaler()
scaler.fit(train[HANDCRAFTED_COLS].fillna(0))

# Transform all splits
scaled_train = scaler.transform(train[HANDCRAFTED_COLS].fillna(0))
scaled_valid = scaler.transform(valid[HANDCRAFTED_COLS].fillna(0))
scaled_test  = scaler.transform(test[HANDCRAFTED_COLS].fillna(0))

# Wrap back into DataFrames with column names
scaled_train_df = pd.DataFrame(scaled_train, columns=[f'sc_{c}' for c in HANDCRAFTED_COLS], index=train.index)
scaled_valid_df = pd.DataFrame(scaled_valid, columns=[f'sc_{c}' for c in HANDCRAFTED_COLS], index=valid.index)
scaled_test_df  = pd.DataFrame(scaled_test,  columns=[f'sc_{c}' for c in HANDCRAFTED_COLS], index=test.index)

print(f'✅ Scaler fitted and applied.')
print(f'   Scaled feature shape: train={scaled_train_df.shape}')

# Quick sanity check — all values should be in [0, 1]
print(f'   Min value: {scaled_train.min():.4f}  Max value: {scaled_train.max():.4f}  (expected [0, 1])')

---
## 📦 Step 10 — Assemble Final Feature Matrices

Now we **combine everything** into one final feature matrix per split.

```
Final Matrix = [  TF-IDF (11,000)  |  Hand-crafted scaled (~80)  ]
```

We use **scipy sparse matrices** because TF-IDF is very sparse (99%+ zeros).  
Storing it as a dense matrix would require gigabytes of memory — sparse keeps it small.

| Block | Size | Format |
|-------|------|--------|
| TF-IDF unigram | 5,000 | sparse |
| TF-IDF bigram | 3,000 | sparse |
| TF-IDF char | 3,000 | sparse |
| Hand-crafted features | ~80 | dense → sparse |
| **Total** | **~11,080** | **sparse** |

In [ ]:
# Stack TF-IDF sparse + dense hand-crafted features
def assemble(tfidf_sparse, dense_df):
    dense_sparse = sp.csr_matrix(dense_df.values)
    return sp.hstack([tfidf_sparse, dense_sparse], format='csr')

X_train = assemble(X_tfidf_train, scaled_train_df)
X_valid = assemble(X_tfidf_valid, scaled_valid_df)
X_test  = assemble(X_tfidf_test,  scaled_test_df)

y_train = train['binary_label_id'].values.astype(int)
y_valid = valid['binary_label_id'].values.astype(int)
y_test  = test['binary_label_id'].values.astype(int)

print('🚀 Final Feature Matrix Summary')
print('=' * 50)
print(f'   X_train : {X_train.shape}   y: {Counter(y_train)}')
print(f'   X_valid : {X_valid.shape}   y: {Counter(y_valid)}')
print(f'   X_test  : {X_test.shape}   y: {Counter(y_test)}')
print()
print(f'   Feature breakdown:')
print(f'     TF-IDF unigram  : {len(vec_unigram.get_feature_names_out()):>6,}')
print(f'     TF-IDF bigram   : {len(vec_bigram.get_feature_names_out()):>6,}')
print(f'     TF-IDF char     : {len(vec_char.get_feature_names_out()):>6,}')
print(f'     Hand-crafted    : {len(HANDCRAFTED_COLS):>6,}')
print(f'     ─────────────────────')
print(f'     TOTAL           : {X_train.shape[1]:>6,}')
print(f'\n   Sparsity (X_train): {1 - X_train.nnz / (X_train.shape[0]*X_train.shape[1]):.1%}')

---
## 📊 Step 11 — Feature Importance Visualisation

Before saving, we get a quick look at which **hand-crafted features** best separate FAKE from REAL.  
We measure this using the **absolute mean difference** between FAKE and REAL distributions — a simple but effective signal of discriminative power.

In [ ]:
# Compute mean difference between FAKE and REAL for each hand-crafted feature
fake_means = train[train['binary_label']=='FAKE'][HANDCRAFTED_COLS].mean()
real_means = train[train['binary_label']=='REAL'][HANDCRAFTED_COLS].mean()
diff       = (fake_means - real_means).abs().sort_values(ascending=False)

top_diff = diff.head(25)

fig, ax = plt.subplots(figsize=(10, 9))
colors_bar = ['#e57373' if (fake_means[c] > real_means[c]) else '#81c784'
              for c in top_diff.index]
bars = ax.barh(top_diff.index[::-1], top_diff.values[::-1],
               color=colors_bar[::-1], edgecolor='white')
ax.set_title('📊 Top 25 Hand-Crafted Features by\nAbsolute Mean Difference (FAKE − REAL)',
             fontweight='bold')
ax.set_xlabel('|Mean(FAKE) − Mean(REAL)|')

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#e57373', label='FAKE > REAL'),
    Patch(color='#81c784', label='REAL > FAKE')
], fontsize=9)

plt.tight_layout()
plt.show()

print('\n💡 Features with the largest FAKE vs REAL difference are the most discriminative.')
print('   Credit history features (cred_*) and frequency encodings dominate — as expected.')

In [ ]:
# Summary pie chart — feature group proportions
feature_groups = {
    'TF-IDF\nUnigram':    len(vec_unigram.get_feature_names_out()),
    'TF-IDF\nBigram':     len(vec_bigram.get_feature_names_out()),
    'TF-IDF\nChar n-gram':len(vec_char.get_feature_names_out()),
    'Linguistic\nFeatures':len(LING_COLS),
    'Readability\nScores': 4,
    'Sentiment\nProxy':    len(SENT_COLS),
    'Credibility\nFeatures':len(CRED_COLS),
    'Categorical\nFreq Enc':len(FREQ_COLS),
    'Other\nHandcrafted':  len(HANDCRAFTED_COLS) - len(LING_COLS) - len(SENT_COLS) - len(CRED_COLS) - len(FREQ_COLS) - 4,
}

colors_pie = ['#42a5f5','#64b5f6','#90caf9',
              '#ef5350','#ab47bc','#ff7043',
              '#26a69a','#ffa726','#78909c']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].pie(list(feature_groups.values()), labels=list(feature_groups.keys()),
            colors=colors_pie, autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor':'white','linewidth':1.5}, textprops={'fontsize':9})
axes[0].set_title('Feature Group Proportions', fontweight='bold')

bars = axes[1].bar(list(feature_groups.keys()), list(feature_groups.values()),
                   color=colors_pie, edgecolor='white')
axes[1].bar_label(bars, padding=3, fontsize=8)
axes[1].set_title('Feature Count per Group', fontweight='bold')
axes[1].set_ylabel('Number of Features')
axes[1].tick_params(axis='x', labelsize=8)

plt.suptitle(f'📦 Final Feature Matrix: {X_train.shape[1]:,} Total Features',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 💾 Step 12 — Save All Outputs

We save everything needed for model training:

| File | Contents | Format |
|------|----------|--------|
| `X_train/valid/test.npz` | Final sparse feature matrices | `.npz` sparse |
| `y_train/valid/test.npy` | Binary label arrays | `.npy` |
| `train/valid/test_features.csv` | Full enriched DataFrames | `.csv` |
| `tfidf_unigram/bigram/char.pkl` | Fitted TF-IDF vectorisers | `.pkl` |
| `scaler.pkl` | Fitted MinMaxScaler | `.pkl` |
| `freq_maps.pkl` | Frequency encoding maps | `.pkl` |
| `feature_names.txt` | Names of all features in order | `.txt` |

The `.pkl` files are crucial — they let you **apply the same transformations** to new/test data during inference, without re-fitting.

In [ ]:
# ── Save sparse matrices ──────────────────────────────────────────────
sp.save_npz(f'{OUTPUT_DIR}/X_train.npz', X_train)
sp.save_npz(f'{OUTPUT_DIR}/X_valid.npz', X_valid)
sp.save_npz(f'{OUTPUT_DIR}/X_test.npz',  X_test)

# ── Save label arrays ─────────────────────────────────────────────────
np.save(f'{OUTPUT_DIR}/y_train.npy', y_train)
np.save(f'{OUTPUT_DIR}/y_valid.npy', y_valid)
np.save(f'{OUTPUT_DIR}/y_test.npy',  y_test)

# ── Save enriched feature CSVs ────────────────────────────────────────
train.to_csv(f'{OUTPUT_DIR}/train_features.csv', index=False)
valid.to_csv(f'{OUTPUT_DIR}/valid_features.csv', index=False)
test.to_csv( f'{OUTPUT_DIR}/test_features.csv',  index=False)

# ── Save fitted transformers ──────────────────────────────────────────
joblib.dump(vec_unigram, f'{OUTPUT_DIR}/tfidf_unigram.pkl')
joblib.dump(vec_bigram,  f'{OUTPUT_DIR}/tfidf_bigram.pkl')
joblib.dump(vec_char,    f'{OUTPUT_DIR}/tfidf_char.pkl')
joblib.dump(scaler,      f'{OUTPUT_DIR}/scaler.pkl')
joblib.dump(freq_maps,   f'{OUTPUT_DIR}/freq_maps.pkl')

# ── Save feature name manifest ────────────────────────────────────────
all_feature_names = (
    [f'tfidf_uni_{w}'  for w in vec_unigram.get_feature_names_out()] +
    [f'tfidf_bi_{w}'   for w in vec_bigram.get_feature_names_out()] +
    [f'tfidf_char_{w}' for w in vec_char.get_feature_names_out()] +
    [f'sc_{c}'         for c in HANDCRAFTED_COLS]
)
with open(f'{OUTPUT_DIR}/feature_names.txt', 'w') as f:
    f.write(f'Total features: {len(all_feature_names)}\n')
    f.write(f'TF-IDF unigram : {len(vec_unigram.get_feature_names_out())}\n')
    f.write(f'TF-IDF bigram  : {len(vec_bigram.get_feature_names_out())}\n')
    f.write(f'TF-IDF char    : {len(vec_char.get_feature_names_out())}\n')
    f.write(f'Hand-crafted   : {len(HANDCRAFTED_COLS)}\n\n')
    f.write('--- Feature names ---\n')
    f.write('\n'.join(all_feature_names))

print('💾 All outputs saved:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{fname}') / 1024
    print(f'   {fname:40s}  {size:8.1f} KB')

---
## ✅ Step 13 — Final Summary

A complete summary of everything extracted and saved.

In [ ]:
print('=' * 65)
print('  LIAR FEATURE EXTRACTION — COMPLETE ✅')
print('=' * 65)
print()
print('📐 FEATURE MATRIX SHAPES:')
print(f'   X_train : {X_train.shape}   labels: {Counter(y_train)}')
print(f'   X_valid : {X_valid.shape}   labels: {Counter(y_valid)}')
print(f'   X_test  : {X_test.shape}   labels: {Counter(y_test)}')
print()
print('🔧 FEATURE GROUPS:')
print(f'   TF-IDF (unigram)         : {len(vec_unigram.get_feature_names_out()):>5,} features')
print(f'   TF-IDF (bigram)          : {len(vec_bigram.get_feature_names_out()):>5,} features')
print(f'   TF-IDF (char n-gram)     : {len(vec_char.get_feature_names_out()):>5,} features')
print(f'   Linguistic features      : {len(LING_COLS):>5,} features')
print(f'   Readability scores       : {4:>5,} features')
print(f'   Sentiment proxies        : {len(SENT_COLS):>5,} features')
print(f'   Credibility features     : {len(CRED_COLS):>5,} features')
print(f'   Categorical freq enc     : {len(FREQ_COLS):>5,} features')
print(f'   Other hand-crafted       : {len(HANDCRAFTED_COLS):>5,} total hand-crafted (incl. above)')
print(f'   ──────────────────────────────────')
print(f'   GRAND TOTAL              : {X_train.shape[1]:>5,} features')
print()
print('📁 SAVED TO:', OUTPUT_DIR)
print()
print('🚀 READY FOR MODEL TRAINING:')
print('   • Logistic Regression   → load X_train.npz + y_train.npy')
print('   • Gradient Boosting     → load X_train.npz + y_train.npy')
print('   • XGBoost / LightGBM    → load X_train.npz + y_train.npy')
print('   • BERT fine-tuning      → use train_features.csv (statement_bert col)')
print()
print('📌 NOTE: For BERT, use statement_bert from the features CSV.')
print('         For TF-IDF + classical ML, load X_train.npz directly.')
print('=' * 65)

---
## 🗺️ What's Next?

### Model Training (next notebook)

| Model | Input | Expected Accuracy |
|-------|-------|------------------|
| **Logistic Regression** | `X_train.npz` (TF-IDF + scaled features) | ~63–66% |
| **Gradient Boosting** | `X_train.npz` or `train_features.csv` hand-crafted cols | ~65–68% |
| **BERT** | `statement_bert` from CSV | ~68–72% |

### Quick-start code for model training:
```python
import scipy.sparse as sp, numpy as np
from sklearn.linear_model import LogisticRegression

X_train = sp.load_npz('feature_store/X_train.npz')
X_valid = sp.load_npz('feature_store/X_valid.npz')
y_train = np.load('feature_store/y_train.npy')
y_valid = np.load('feature_store/y_valid.npy')

model = LogisticRegression(C=1.0, max_iter=1000, solver='saga')
model.fit(X_train, y_train)
print('Valid accuracy:', model.score(X_valid, y_valid))
```

---
*Dataset: LIAR — William Yang Wang, ACL 2017 | Project #13: Fake News Detection Using NLP Classifiers*